In [ ]:
import geopandas as gpd
import pandas as pd
import os

# load the geojson for 0-5 cm
cfr_0_5 = gpd.read_file("D:/tierra/data/soil/cfr/cfr_05cm.geojson")
# load the geojson for 5-15 cm
cfr_5_15 = gpd.read_file("D:/tierra/data/soil/cfr/cfr_15cm.geojson")
# load the geojson for 15-30 cm
cfr_15_30 = gpd.read_file("D:/tierra/data/soil/cfr/cfr_30cm.geojson")

In [2]:
cfr_0_5.head()

,fid,DN,_mean,geometry
0,1.0,3,3.350253,"MULTIPOLYGON (((-114.72558 32.72561, -114.7159..."
1,2.0,3,3.426273,"MULTIPOLYGON (((-114.85062 32.71599, -114.841 ..."
2,3.0,6,5.800558,"MULTIPOLYGON (((-114.841 32.71599, -114.83138 ..."
3,4.0,7,7.194875,"MULTIPOLYGON (((-114.83138 32.71599, -114.8217..."
4,5.0,5,4.980327,"MULTIPOLYGON (((-114.81215 32.71599, -114.8025..."


In [8]:
cfr_5_15.head()

,fid,DN,_count,_sum,_mean,geometry
0,1,2,1.0,1.861116,1.861116,"POLYGON ((-114.72558 32.72561, -114.72558 32.7..."
1,2,6,1.0,5.945132,5.945132,"POLYGON ((-114.841 32.71599, -114.841 32.70637..."
2,3,7,1.0,7.083116,7.083116,"POLYGON ((-114.83138 32.71599, -114.83138 32.7..."
3,4,5,1.0,4.773536,4.773536,"POLYGON ((-114.81215 32.71599, -114.81215 32.7..."
4,5,8,2.0,16.574183,8.287092,"POLYGON ((-114.76405 32.71599, -114.76405 32.7..."


In [9]:
cfr_15_30.head()

,fid,DN,cfr_30cmcount,cfr_30cmsum,cfr_30cmmean,geometry
0,1,2,1.0,2.164754,2.164754,"POLYGON ((-114.72558 32.72561, -114.72558 32.7..."
1,2,4,1.0,3.922217,3.922217,"POLYGON ((-114.841 32.71599, -114.841 32.70637..."
2,3,6,1.0,6.123470,6.123470,"POLYGON ((-114.83138 32.71599, -114.83138 32.7..."
3,4,4,1.0,3.706957,3.706957,"POLYGON ((-114.77367 32.71599, -114.77367 32.7..."
4,5,7,2.0,13.012083,6.506041,"POLYGON ((-114.76405 32.71599, -114.76405 32.7..."


In [17]:
target = 'tceq'
base_path = f"D:/tierra/outputs/unfiltered/harmonized/"
input_file = os.path.join(base_path, f"Mexico_standardized_cleaned_{target}.csv")
output_file = os.path.join(base_path, f"Mexico_standardized_cfvo_{target}.csv")

wosis = pd.read_csv(input_file)

# drop geometry column
wosis = wosis.drop(columns=['geometry']).copy()

# convert wosis to geodataframe using lat/lon
wosis = gpd.GeoDataFrame(wosis, geometry=gpd.points_from_xy(wosis['longitude'], wosis['latitude']), crs="EPSG:4326")

wosis.head()

,profile_id,depth_category,date,longitude,latitude,clay,phaq,sand,silt,tceq,soil_type,slopemean,bedrock,geometry
0,1149115,0_5,2004-5-27,-92.924872,15.375747,11.473293,5.917957,53.411810,35.114897,0.0,FLUVISOL EUTRICO,0.153699,rocas_metamorficas,POINT (-92.92487 15.37575)
1,1149115,5_15,2004-5-27,-92.924872,15.375747,11.914496,5.902915,48.878541,39.206964,0.0,FLUVISOL EUTRICO,0.153699,rocas_metamorficas,POINT (-92.92487 15.37575)
2,1149115,15_30,2004-5-27,-92.924872,15.375747,13.443150,5.854219,33.299825,53.257025,0.0,FLUVISOL EUTRICO,0.153699,rocas_metamorficas,POINT (-92.92487 15.37575)
3,1149116,0_5,2004-11-12,-93.805531,15.933933,10.280596,5.814359,83.221049,6.498355,0.0,REGOSOL EUTRICO,0.153699,rocas_metamorficas,POINT (-93.80553 15.93393)
4,1149116,5_15,2004-11-12,-93.805531,15.933933,9.732159,5.981748,84.743544,5.524297,0.0,REGOSOL EUTRICO,0.153699,rocas_metamorficas,POINT (-93.80553 15.93393)


In [18]:
# Spatial join between wosis and cfr_0_5 for depth 0-5cm
wosis_0_5 = wosis[wosis['depth_category'] == '0_5'].copy()
wosis_0_5 = gpd.sjoin(wosis_0_5, cfr_0_5[['_mean', 'geometry']], how='left', predicate='within')
wosis_0_5 = wosis_0_5.rename(columns={'_mean': 'cfr'})
wosis_0_5 = wosis_0_5.drop(columns=['index_right'])

# Spatial join between wosis and cfr_5_15 for depth 5-15cm
wosis_5_15 = wosis[wosis['depth_category'] == '5_15'].copy()
wosis_5_15 = gpd.sjoin(wosis_5_15, cfr_5_15[['_mean', 'geometry']], how='left', predicate='within')
wosis_5_15 = wosis_5_15.rename(columns={'_mean': 'cfr'})
wosis_5_15 = wosis_5_15.drop(columns=['index_right'])

# Spatial join between wosis and cfr_15_30 for depth 15-30cm
wosis_15_30 = wosis[wosis['depth_category'] == '15_30'].copy()
wosis_15_30 = gpd.sjoin(wosis_15_30, cfr_15_30[['cfr_30cmmean', 'geometry']], how='left', predicate='within')
wosis_15_30 = wosis_15_30.rename(columns={'cfr_30cmmean': 'cfr'})
wosis_15_30 = wosis_15_30.drop(columns=['index_right'])

# Concatenate the three dataframes
wosis_combined = pd.concat([wosis_0_5, wosis_5_15, wosis_15_30], ignore_index=True)

print(f"Shape of the combined dataframe: {wosis_combined.shape}")
# Check for missing values in the 'cfr' column
print(f"Missing values in 'cfr' column: {wosis_combined['cfr'].isnull().sum()}")
wosis_combined.head()

Shape of the combined dataframe: (6498, 15)
Missing values in 'cfr' column: 15


,profile_id,depth_category,date,longitude,latitude,clay,phaq,sand,silt,tceq,soil_type,slopemean,bedrock,geometry,cfr
0,1149115,0_5,2004-5-27,-92.924872,15.375747,11.473293,5.917957,53.411810,35.114897,0.0,FLUVISOL EUTRICO,0.153699,rocas_metamorficas,POINT (-92.92487 15.37575),0.001939
1,1149116,0_5,2004-11-12,-93.805531,15.933933,10.280596,5.814359,83.221049,6.498355,0.0,REGOSOL EUTRICO,0.153699,rocas_metamorficas,POINT (-93.80553 15.93393),0.001939
2,1149117,0_5,1984-4-15,-92.575732,14.871044,2.000000,5.779616,93.864105,4.135895,0.0,REGOSOL EUTRICO,0.153699,rocas_igneas,POINT (-92.57573 14.87104),NaN
3,1149118,0_5,1984-4-15,-92.340858,14.761703,18.847018,6.132662,50.105555,31.047427,0.0,CAMBISOL EUTRICO,0.153699,rocas_igneas,POINT (-92.34086 14.7617),0.001939
4,1149120,0_5,1990-7-12,-104.979876,19.941109,10.430778,4.771774,69.687338,19.881883,0.0,CAMBISOL CROMICO,2.462867,rocas_igneas,POINT (-104.97988 19.94111),2.349696


In [19]:
# drop missing values
wosis_combined = wosis_combined.dropna()
print(f"Shape of the combined dataframe after dropping missing values: {wosis_combined.shape}")

# drop duplicate rows
wosis_combined = wosis_combined.drop_duplicates()
print(f"Shape of the combined dataframe after dropping duplicates: {wosis_combined.shape}")

# drop geometry column
wosis_combined = wosis_combined.drop(columns=['geometry']).copy()

# reset index order by profile_id, depth_category
wosis_combined = wosis_combined.sort_values(by=['profile_id', 'depth_category']).reset_index(drop=True)

# save the combined dataframe to a new csv file
wosis_combined.to_csv(output_file, index=False)
print(f"Combined dataframe saved to {output_file}")

wosis_combined.head()

Shape of the combined dataframe after dropping missing values: (6481, 15)
Shape of the combined dataframe after dropping duplicates: (5635, 15)
Combined dataframe saved to D:/tierra/outputs/unfiltered/harmonized/Mexico_standardized_cfvo_tceq.csv


,profile_id,depth_category,date,longitude,latitude,clay,phaq,sand,silt,tceq,soil_type,slopemean,bedrock,cfr
0,1149115,0_5,2004-5-27,-92.924872,15.375747,11.473293,5.917957,53.411810,35.114897,0.0,FLUVISOL EUTRICO,0.153699,rocas_metamorficas,0.001939
1,1149115,15_30,2004-5-27,-92.924872,15.375747,13.443150,5.854219,33.299825,53.257025,0.0,FLUVISOL EUTRICO,0.153699,rocas_metamorficas,0.011414
2,1149115,5_15,2004-5-27,-92.924872,15.375747,11.914496,5.902915,48.878541,39.206964,0.0,FLUVISOL EUTRICO,0.153699,rocas_metamorficas,0.000931
3,1149116,0_5,2004-11-12,-93.805531,15.933933,10.280596,5.814359,83.221049,6.498355,0.0,REGOSOL EUTRICO,0.153699,rocas_metamorficas,0.001939
4,1149116,15_30,2004-11-12,-93.805531,15.933933,8.312555,6.448620,88.852404,2.835040,0.0,REGOSOL EUTRICO,0.153699,rocas_metamorficas,0.011414
